# IOAI — 2025 Stage 2 Source Extraction (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
if not os.path.exists('data/corpus.jsonl'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-2-source-extraction/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 소스 추출 — 밀집 검색 모범답안 (SGPT dense retrieval)

폴란드 AI 올림피아드 II · 2025 · 2단계. 과학 주장 → 근거 문서 검색(SciFact 계열). **SGPT-125M**
(GPT 기반 문장 임베더, weighted-mean pooling + special brackets)로 쿼리/문서를 임베딩해 코사인 상위10 검색.

**성능(valid 300쿼리·corpus 5183, 실측)**: nDCG **0.496** → **99/100**.
(베이스라인=미구현 임베더 0점. *참고*: 순수 TF-IDF 어휘검색도 nDCG≈0.54로 만점권 — 이 과제는 채점
상한이 0.5라 어휘검색만으로도 통과되나, 의도된 기법은 **신경 밀집 검색**이라 그대로 재현한다.)

**제출**: `submission.csv` — `query_id,doc_ids` (상위10 text_id 랭크순 공백구분).


In [ ]:
# 데이터 준비 (Colab: 자동 다운로드 / DGX: data/ 이미 존재)
import os, urllib.request, zipfile
if not os.path.exists("data/corpus.jsonl"):
    url = "https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-2-source-extraction/data.zip"
    urllib.request.urlretrieve(url, "d.zip"); zipfile.ZipFile("d.zip").extractall("data")

import json, csv, torch
from transformers import AutoModel, AutoTokenizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Tokenizer:
    def __init__(self, tokenizer_path, length=150):
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "right"
        self.length = length
    def __call__(self, batch_text):
        return self.tokenizer(batch_text, max_length=self.length, truncation=True,
                              padding=True, return_tensors="pt").to(device)

def load_corpus(file):
    corpus = {}
    for line in open(file, encoding="utf8"):
        o = json.loads(line); corpus[o["text_id"]] = {"text": o.get("text"), "title": o.get("title")}
    return corpus

def load_queries(file):                       # 이 사이트판: query_id + query (정답 미포함)
    queries = {}
    for line in open(file, encoding="utf8"):
        o = json.loads(line); queries[o["query_id"]] = o["query"]
    return queries

corpus = load_corpus("data/corpus.jsonl")
queries = load_queries("data/queries.jsonl")
print(f"corpus {len(corpus)} texts, queries {len(queries)}")


In [ ]:
# 검색(고정 코드) — 코사인 유사도 상위 k
def cos_sim(a, b):
    a = torch.nn.functional.normalize(a, p=2, dim=1)
    b = torch.nn.functional.normalize(b, p=2, dim=1)
    return torch.mm(a, b.transpose(0, 1))

def search_topk_texts(embedder, corpus, queries, top_k=10):
    query_ids = list(queries.keys())
    q_texts = [queries[q] for q in query_ids]
    query_embeddings = embedder.encode_queries(q_texts)
    corpus_ids = list(corpus.keys())
    corpus_texts = [corpus[c] for c in corpus_ids]
    corpus_embeddings = embedder.encode_corpus(corpus_texts)
    cos = cos_sim(query_embeddings, corpus_embeddings)
    cos[torch.isnan(cos)] = -1
    top_val, top_idx = torch.topk(cos, min(top_k, cos.shape[1]), dim=1, largest=True, sorted=True)
    results = {}
    for qi, qid in enumerate(query_ids):
        results[qid] = [corpus_ids[j] for j in top_idx[qi].cpu().tolist()]
    return results

def save_submission(results, path="submission.csv"):
    with open(path, "w", newline="") as f:
        w = csv.writer(f); w.writerow(["query_id", "doc_ids"])
        for qid, ids in results.items():
            w.writerow([qid, " ".join(str(x) for x in ids)])
    print("submission.csv 저장:", len(results), "쿼리")


In [ ]:
class Embedder:
    """SGPT-125M 밀집 임베더: weighted-mean pooling + special brackets(쿼리 [ ], 문서 { }).
    쿼리와 문서를 같은 공간에 임베딩해 코사인 유사도로 검색한다."""
    MODEL = "Muennighoff/SGPT-125M-weightedmean-msmarco-specb-bitfit"
    def __init__(self):
        self.tok = AutoTokenizer.from_pretrained(self.MODEL); self.tok.pad_token = self.tok.eos_token
        self.model = AutoModel.from_pretrained(self.MODEL).to(device).eval()
        self.QB = self.tok.encode("[", add_special_tokens=False)[0]; self.QE = self.tok.encode("]", add_special_tokens=False)[0]
        self.DB = self.tok.encode("{", add_special_tokens=False)[0]; self.DE = self.tok.encode("}", add_special_tokens=False)[0]

    def _batch(self, texts, is_query):
        b = self.tok(texts, padding=False, truncation=True, max_length=150)
        ids, att = [], []
        for seq, a in zip(b["input_ids"], b["attention_mask"]):
            bos, eos = (self.QB, self.QE) if is_query else (self.DB, self.DE)
            ids.append([bos] + seq + [eos]); att.append([1] + a + [1])
        m = max(len(s) for s in ids)
        for i in range(len(ids)):
            p = m - len(ids[i]); ids[i] += [self.tok.pad_token_id]*p; att[i] += [0]*p
        return {"input_ids": torch.tensor(ids).to(device), "attention_mask": torch.tensor(att).to(device)}

    @torch.no_grad()
    def _encode(self, texts, is_query, bs=32):
        out = []
        for i in range(0, len(texts), bs):
            b = self._batch(texts[i:i+bs], is_query)
            last = self.model(**b).last_hidden_state
            w = torch.arange(1, last.shape[1]+1, device=device).float()[None, :, None].expand(last.size())
            mask = b["attention_mask"][..., None].float()
            out.append(((last * mask * w).sum(1) / (mask * w).sum(1)))
        return torch.cat(out)

    def encode_queries(self, queries):
        return self._encode(list(queries), True)
    def encode_corpus(self, texts):
        return self._encode([t["title"] + " " + t["text"] for t in texts], False)

embedder = Embedder()
with torch.no_grad():
    results = search_topk_texts(embedder, corpus, queries, top_k=10)
save_submission(results)


### 정리
- SGPT-125M(weighted-mean + specb) 밀집 검색 → nDCG ≈ 0.496 → 99점 (베이스라인 미구현 0점).
- **더 끌어올리려면**: 더 큰 임베더(SGPT-1.3B/5.8B)·재순위(reranker)·TF-IDF 하이브리드.
  (여기서는 원문제 스캐폴드의 Embedder 구조를 충실히 재현하는 데 초점.)


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)